# Notebook 03 — Feature Extraction
**Sugarcane Leaf Disease Detection using Deep Feature Fusion**

This notebook covers:
- MobileNetV2 deep feature extraction (1,280-dim embeddings)
- GLCM texture + HSV colour handcrafted features (11-dim)
- Feature fusion (1,291-dim vectors)
- Saving fused feature matrices for model training


In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
from model import build_feature_extractor, extract_deep_features, fuse_features
from feature_extraction import batch_extract_handcrafted


In [ ]:
# ── Load splits ───────────────────────────────────────────────────────────
X_train = np.load('../data/X_train_paths.npy', allow_pickle=True).tolist()
X_val   = np.load('../data/X_val_paths.npy',   allow_pickle=True).tolist()
X_test  = np.load('../data/X_test_paths.npy',  allow_pickle=True).tolist()

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')


In [ ]:
# ── MobileNetV2 deep features ─────────────────────────────────────────────
extractor = build_feature_extractor()
print('Extracting deep features (train)...')
deep_train = extract_deep_features(extractor, X_train, batch_size=32)
deep_val   = extract_deep_features(extractor, X_val,   batch_size=32)
deep_test  = extract_deep_features(extractor, X_test,  batch_size=32)
print(f'Deep feature shape (train): {deep_train.shape}')  # (N, 1280)


In [ ]:
# ── Handcrafted features ──────────────────────────────────────────────────
print('Extracting handcrafted features (train)...')
hc_train = batch_extract_handcrafted(X_train)
hc_val   = batch_extract_handcrafted(X_val)
hc_test  = batch_extract_handcrafted(X_test)
print(f'Handcrafted feature shape (train): {hc_train.shape}')  # (N, 11)


In [ ]:
# ── Feature fusion ────────────────────────────────────────────────────────
X_train_fused = fuse_features(deep_train, hc_train)
X_val_fused   = fuse_features(deep_val,   hc_val)
X_test_fused  = fuse_features(deep_test,  hc_test)
print(f'Fused feature shape (train): {X_train_fused.shape}')  # (N, 1291)

np.save('../data/X_train_fused.npy', X_train_fused)
np.save('../data/X_val_fused.npy',   X_val_fused)
np.save('../data/X_test_fused.npy',  X_test_fused)
print('Fused feature matrices saved.')
